In [1]:
import glob
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
indirs = [
    "/grid/siepel/home/staklins/projects/crispr_barcode/data/uniform_50cells_50sites_0.0025mut_10-6mig_data_8_22_24",
    "/grid/siepel/home/staklins/projects/crispr_barcode/data/heritable_silencing_001_11_16_25_uniform_50cells_50sites_0.0025mut_10-6mig_data_from_8_22_24",
    "/grid/siepel/home/staklins/projects/crispr_barcode/data/heritable_silencing_005_11_16_25_uniform_50cells_50sites_0.0025mut_10-6mig_data_from_8_22_24",
    "/grid/siepel/home/staklins/projects/crispr_barcode/data/heritable_silencing_01_11_16_25_uniform_50cells_50sites_0.0025mut_10-6mig_data_from_8_22_24"
]

outfile="/grid/siepel/home/staklins/projects/crispr_barcode/results/beam/latest_results/heritable_silencing_01_11_16_25_uniform_50cells_50sites_0.0025mut_10-6mig_data_from_8_22_24/missing_data_proportions.pdf"

# Match order of this list to indirs
heritable_rates = ["0.0001", "0.001", "0.005", "0.01"]

indel_files = {}
for heritable_rate, indir in zip(heritable_rates, indirs):
    indel_files[heritable_rate] = glob.glob(f"{indir}/*/*_indel_character_matrix.tsv")
    print(f"Heritable silencing rate: {heritable_rate}")
    print(f"Number of indel character matrices: {len(indel_files[heritable_rate])}")



Heritable silencing rate: 0.0001
Number of indel character matrices: 100
Heritable silencing rate: 0.001
Number of indel character matrices: 100
Heritable silencing rate: 0.005
Number of indel character matrices: 100
Heritable silencing rate: 0.01
Number of indel character matrices: 100


In [3]:
def calculate_missing_data_proportion(barcode_df):
    total_entries = barcode_df.size
    missing_entries = (barcode_df == -1).sum().sum()
    return missing_entries / total_entries


In [4]:
missing_data_proportions = {}
for heritable_rate in heritable_rates:
    missing_data_proportions[heritable_rate] = []
    for indel_file in indel_files[heritable_rate]:
        barcode_df = pd.read_csv(indel_file, sep="\t", index_col=0)
        proportion_missing = calculate_missing_data_proportion(barcode_df)
        missing_data_proportions[heritable_rate].append(proportion_missing)

In [11]:
rows = []
for rate, vals in missing_data_proportions.items():
    for v in vals:
        rows.append({"heritable_rate": rate, "missing_prop": v})

df = pd.DataFrame(rows)

# Font sizes
fs = 24

plt.figure(figsize=(8, 6))

palette = sns.color_palette("colorblind", n_colors=df["heritable_rate"].nunique())

# --------------------------
# Seaborn barplot: mean + SD
# --------------------------
ax = sns.barplot(
    data=df,
    x="heritable_rate",
    y="missing_prop",
    palette=palette,
    errorbar="sd",
    edgecolor="black",
    linewidth=1,
    capsize=0.2
)

# # Optional: overlay jittered points
# sns.stripplot(
#     data=df,
#     x="heritable_rate",
#     y="missing_prop",
#     color="black",
#     size=6,
#     alpha=0.3,
#     jitter=True
# )

# Labels and formatting
ax.set_xlabel("Heritable silencing rate", fontsize=fs)
ax.set_ylabel("Missing data proportion", fontsize=fs)
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelsize=fs-4)
ax.tick_params(axis="y", labelsize=fs-4)

# Add border around plot
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)

plt.tight_layout()
plt.savefig(outfile)
# plt.show()
plt.close()


/tmp/ipykernel_2938862/462999997.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(
